# Step 4: RAG Query Interface

Ask questions about the Harry Potter books. Uses hybrid search (keyword + vector) with semantic reranking on Azure AI Search, then sends the top results to GPT-4.1 for answer generation with page citations.

In [1]:
%pip install openai azure-identity azure-search-documents python-dotenv -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery, QueryType
from dotenv import load_dotenv

load_dotenv(override=True)

# Auth
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

# Azure OpenAI
openai_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_ad_token_provider=token_provider,
    api_version="2024-12-01-preview",
)
CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
EMBEDDING_DIMS = 256  # Must match what was used in Step 3

# Azure AI Search
search_client = SearchClient(
    endpoint=os.getenv("AZURE_SEARCH_ENDPOINT"),
    index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "rag-index"),
    credential=DefaultAzureCredential(),
)

print(f"Chat model: {CHAT_DEPLOYMENT}")
print(f"Embedding model: {EMBEDDING_DEPLOYMENT} (dims={EMBEDDING_DIMS})")
print(f"Search index: {os.getenv('AZURE_SEARCH_INDEX_NAME')}")
print("Ready!")

Chat model: gpt-4.1
Embedding model: text-embedding-3-large-460208 (dims=256)
Search index: rag-index
Ready!


In [2]:
def search(query: str, top_k: int = 5) -> list[dict]:
    """Hybrid search + semantic reranking. Returns top_k chunks with scores."""
    # Embed the query
    response = openai_client.embeddings.create(
        model=EMBEDDING_DEPLOYMENT,
        input=[query],
        dimensions=EMBEDDING_DIMS,
    )
    query_vector = response.data[0].embedding

    results = search_client.search(
        search_text=query,
        vector_queries=[
            VectorizedQuery(vector=query_vector, k=top_k, fields="embedding")
        ],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="default-semantic",
        top=top_k,
    )

    chunks = []
    for r in results:
        chunks.append({
            "content": r["content"],
            "page_number": r["page_number"],
            "score": r["@search.score"],
            "reranker_score": r.get("@search.reranker_score"),
        })
    return chunks

# Quick test
test_results = search("What is the Philosopher's Stone?")
for r in test_results:
    print(f"  Page {r['page_number']} | Reranker: {r['reranker_score']:.2f} | {r['content'][:100]}...")

  Page 50 | Reranker: 2.63 | [Image descriptions]
The image shows a large, imposing figure standing confidently. The person is we...
  Page 257 | Reranker: 2.49 | [Image descriptions]
The image depicts a man wearing dark, flowing robes. He is lifting up his turba...
  Page 198 | Reranker: 2.08 | “Oh, honestly, don’t you two read? Look — read that, there.”
She pushed the book toward them, and Ha...
  Page 3279 | Reranker: 1.94 | letter may be seen on page 463.)
Gellert —
Your point about Wizard dominance being FOR THE MUGGLES’ ...
  Page 267 | Reranker: 1.91 | “Yes, him — Quirrell said he hates me because he hated my father. Is that
true?”
“Well, they did rat...


In [3]:
def ask(question: str, top_k: int = 5) -> str:
    """Full RAG pipeline: search → build context → generate answer."""
    # 1. Retrieve relevant chunks
    chunks = search(question, top_k=top_k)

    # 2. Build context from retrieved chunks
    context_parts = []
    for c in chunks:
        context_parts.append(f"[Page {c['page_number']}]\n{c['content']}")
    context = "\n\n---\n\n".join(context_parts)

    # 3. Generate answer with GPT-4.1
    response = openai_client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant that answers questions about the Harry Potter books. "
                    "Use ONLY the provided context to answer. If the context doesn't contain enough "
                    "information, say so. Always cite page numbers in your answer like (Page X)."
                ),
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}",
            },
        ],
        temperature=0.3,
        max_tokens=1000,
    )

    answer = response.choices[0].message.content

    # 4. Print results
    print(f"Question: {question}\n")
    print(f"Answer:\n{answer}\n")
    print(f"Sources: {', '.join('Page ' + str(c['page_number']) for c in chunks)}")
    return answer

In [5]:
# --- Try it out ---
ask("whats are the different names that Dark Lord had in the book? tell me in which parts of the book these names comes as well")

Question: whats are the different names that Dark Lord had in the book? tell me in which parts of the book these names comes as well

Answer:
Based on the provided context, the main alternative name for the Dark Lord mentioned is "the Dark Lord" itself, which is used as a title for Lord Voldemort. For example, on page 2441, Snape refers to Voldemort as "the Dark Lord" multiple times:

- “the Dark Lord acknowledges it. I am pleased to say, however, that Dumbledore is growing old. The duel with the Dark Lord last month shook him..." (Page 2441)

Additionally, on page 1440, Karkaroff refers to Voldemort as "the Dark Lord" when talking about his supporters:

- “...doing his bidding. I give this information as a sign that I fully and totally renounce him, and am filled with a remorse so deep I can barely —”
“These names are?” said Mr. Crouch sharply.
Karkaroff drew a deep breath.
“There was Antonin Dolohov,” he said. “I — I saw him torture countless Muggles and — and non-supporters of the D

'Based on the provided context, the main alternative name for the Dark Lord mentioned is "the Dark Lord" itself, which is used as a title for Lord Voldemort. For example, on page 2441, Snape refers to Voldemort as "the Dark Lord" multiple times:\n\n- “the Dark Lord acknowledges it. I am pleased to say, however, that Dumbledore is growing old. The duel with the Dark Lord last month shook him..." (Page 2441)\n\nAdditionally, on page 1440, Karkaroff refers to Voldemort as "the Dark Lord" when talking about his supporters:\n\n- “...doing his bidding. I give this information as a sign that I fully and totally renounce him, and am filled with a remorse so deep I can barely —”\n“These names are?” said Mr. Crouch sharply.\nKarkaroff drew a deep breath.\n“There was Antonin Dolohov,” he said. “I — I saw him torture countless Muggles and — and non-supporters of the Dark Lord.” (Page 1440)\n\nNo other names for Voldemort (such as "He-Who-Must-Not-Be-Named" or "You-Know-Who") are mentioned in the p

In [6]:
ask("in chamber of secrets, how is voldemort's name and how he names himself?")

Question: in chamber of secrets, how is voldemort's name and how he names himself?

Answer:
In "Harry Potter and the Chamber of Secrets," Voldemort reveals to Harry how he created his name. His birth name is Tom Marvolo Riddle, and he rearranges the letters to spell "I am Lord Voldemort." He explains that he did not want to keep his "filthy Muggle father's name" and instead fashioned a new name that wizards would fear to speak (Page 542).

Sources: Page 542, Page 3507, Page 2952, Page 430, Page 526


'In "Harry Potter and the Chamber of Secrets," Voldemort reveals to Harry how he created his name. His birth name is Tom Marvolo Riddle, and he rearranges the letters to spell "I am Lord Voldemort." He explains that he did not want to keep his "filthy Muggle father\'s name" and instead fashioned a new name that wizards would fear to speak (Page 542).'

In [7]:
# Ask your own question
ask("who si harrys god father and how is he related to his pareents?")

Question: who si harrys god father and how is he related to his pareents?

Answer:
Harry's godfather is Sirius Black. He was Harry's mum and dad's best friend (Page 939).

Sources: Page 939, Page 3018, Page 2194, Page 3195, Page 3416


"Harry's godfather is Sirius Black. He was Harry's mum and dad's best friend (Page 939)."